# 04 KNeighborsClassifier Model Selection

In diesem Notebook wird der KNeighborsClassifier mit Group Cross-Validation nach `subject` bewertet. Anschließend wird das beste kNN-Modell auf dem festgelegten Testset mit den Subjects 27 bis 30 ausgewertet.

In [ ]:
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, cross_validate
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

PROCESSED_DIR = Path("../../data/processed")
RESULTS_DIR = Path("../../results")
TABLE_DIR = RESULTS_DIR / "tables"
FIGURE_DIR = RESULTS_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## Daten laden

In [ ]:
train_data = pd.read_csv(PROCESSED_DIR / "train_data.csv")
test_data = pd.read_csv(PROCESSED_DIR / "test_data.csv")

with open(PROCESSED_DIR / "feature_columns.txt", "r", encoding="utf-8") as f:
    feature_cols = [line.strip() for line in f]

print("Train data:", train_data.shape)
print("Test data:", test_data.shape)
print("Anzahl Features:", len(feature_cols))

## Features, Zielvariable und Groups definieren

In [ ]:
target_col = "activity"
group_col = "subject"

X_train = train_data[feature_cols]
y_train = train_data[target_col]
groups_train = train_data[group_col]

X_test = test_data[feature_cols]
y_test = test_data[target_col]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("groups_train:", groups_train.shape)

print("\nX_test:", X_test.shape)
print("y_test:", y_test.shape)

## kNN-Pipeline definieren

kNN basiert auf Abständen zwischen Beobachtungen. Deshalb wird das Modell mit `StandardScaler` in einer Pipeline kombiniert.

In [ ]:
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

knn_pipeline

## Cross-Validation und Hyperparameter-Suche

Die Modellwahl erfolgt mit 5-fold Group Cross-Validation. Die Gruppierung erfolgt nach `subject`, damit Messungen derselben Person nicht gleichzeitig in Trainings- und Validierungsdaten vorkommen.

In [ ]:
cv = GroupKFold(n_splits=5)

knn_param_distributions = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15, 21],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"],
}

In [ ]:
knn_random_search = RandomizedSearchCV(
    estimator=knn_pipeline,
    param_distributions=knn_param_distributions,
    n_iter=10,
    scoring="accuracy",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
)

knn_random_search.fit(
    X_train,
    y_train,
    groups=groups_train,
)

## Beste Parameter und Cross-Validation-Ergebnis

In [ ]:
print("Beste Parameter:")
print(knn_random_search.best_params_)

print("\nBeste CV Accuracy:")
print(knn_random_search.best_score_)

print("\nBester CV Error:")
print(1 - knn_random_search.best_score_)

## Suchergebnisse als Tabelle

In [ ]:
knn_search_results = pd.DataFrame(knn_random_search.cv_results_)

selected_columns = [
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "std_train_score",
    "mean_fit_time",
    "mean_score_time",
    "params",
]

knn_search_results_table = knn_search_results[selected_columns].sort_values(
    by="mean_test_score",
    ascending=False,
)

knn_search_results_table.head(10)

## Ergebnisse speichern

In [ ]:
knn_search_results_table.to_csv(
    TABLE_DIR / "knn_random_search_results.csv",
    index=False,
)

print("Gespeichert:", TABLE_DIR / "knn_random_search_results.csv")

## Bestes kNN-Modell erneut mit Cross-Validation auswerten

In [ ]:
best_knn_model = knn_random_search.best_estimator_

knn_cv_results = cross_validate(
    best_knn_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=-1,
)

knn_summary = pd.DataFrame({
    "model": ["KNeighborsClassifier_tuned"],
    "mean_cv_accuracy": [knn_cv_results["test_score"].mean()],
    "std_cv_accuracy": [knn_cv_results["test_score"].std()],
    "mean_cv_error": [1 - knn_cv_results["test_score"].mean()],
    "mean_train_accuracy": [knn_cv_results["train_score"].mean()],
    "mean_fit_time": [knn_cv_results["fit_time"].mean()],
    "mean_score_time": [knn_cv_results["score_time"].mean()],
})

knn_summary

## CV-Zusammenfassung speichern

In [ ]:
knn_summary.to_csv(
    TABLE_DIR / "knn_tuned_cv_summary.csv",
    index=False,
)

print("Gespeichert:", TABLE_DIR / "knn_tuned_cv_summary.csv")

## Bewertung auf dem Testset

Nach der Modellwahl wird das beste kNN-Modell auf den Trainingsdaten trainiert und anschließend auf dem festgelegten Testset mit den Subjects 27 bis 30 bewertet.

In [ ]:
best_knn_model.fit(X_train, y_train)

y_pred_knn = best_knn_model.predict(X_test)

knn_test_accuracy = accuracy_score(y_test, y_pred_knn)
knn_test_error = 1 - knn_test_accuracy

print("Test Accuracy:", round(knn_test_accuracy, 4))
print("Test Error:", round(knn_test_error, 4))

## Classification Report

In [ ]:
knn_report = classification_report(
    y_test,
    y_pred_knn,
    output_dict=True,
)

knn_report_df = pd.DataFrame(knn_report).transpose()
knn_report_df

In [ ]:
knn_report_df.to_csv(
    TABLE_DIR / "knn_tuned_classification_report.csv",
    index=True,
)

print("Gespeichert:", TABLE_DIR / "knn_tuned_classification_report.csv")

## Confusion Matrix

Die Confusion Matrix zeigt, welche echten Klassen welchen vorhergesagten Klassen zugeordnet wurden.

In [ ]:
labels = sorted(y_train.unique())

knn_cm = confusion_matrix(
    y_test,
    y_pred_knn,
    labels=labels,
)

knn_cm_table = pd.DataFrame(
    knn_cm,
    index=[f"True: {label}" for label in labels],
    columns=[f"Pred: {label}" for label in labels],
)

knn_cm_table

In [ ]:
knn_cm_table.to_csv(
    TABLE_DIR / "knn_tuned_confusion_matrix.csv",
    index=True,
)

print("Gespeichert:", TABLE_DIR / "knn_tuned_confusion_matrix.csv")

In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=knn_cm,
    display_labels=labels,
)

fig, ax = plt.subplots(figsize=(9, 7))

disp.plot(
    ax=ax,
    xticks_rotation=45,
    values_format="d",
)

plt.title("Confusion Matrix für getunten KNeighborsClassifier")
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "knn_tuned_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## Testmetriken speichern

In [ ]:
knn_test_metrics = pd.DataFrame({
    "model": ["KNeighborsClassifier_tuned"],
    "test_accuracy": [knn_test_accuracy],
    "test_error": [knn_test_error],
    "n_test_observations": [len(y_test)],
})

knn_test_metrics.to_csv(
    TABLE_DIR / "knn_tuned_test_metrics.csv",
    index=False,
)

knn_test_metrics